Paquetes necesarios

Podremos hacer uso del mismo *environment* de la primera práctica, aunque en ocasiones pedirá instalar Pillow

In [ ]:
import cv2  
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

Carga imagen desde archivo con OpenCV y convierte a RGB

In [ ]:
#Lee imagen de archivo
img = cv2.imread('mandril.jpg') 

#Si hay lectura correcta
if img is not None:
    #Muestra dimensiones
    print(img.shape)
    #Muestra la imagen original con matplotlib
    plt.figure()
    #Elimina etiquetas de los ejes que muestra matplotlib
    plt.axis("off")
    plt.imshow(img) 
    plt.show()

    #Recordar que OpenCV lee las imágenes almacenando en formato BGR, por lo que convertimos para visualizr de forma correcta a RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    #Muestra la imagen tras convertir a RGB
    #Eliminamos etiquetas de los ejes
    plt.figure()
    plt.axis("off")
    plt.imshow(img_rgb) 
    plt.show()
else: 
    print('Imagen no encontrada')

Escribe un texto en una posición de la imagen

In [ ]:
# Define tipografías (Bastante limitadas en OpenCV)
font = cv2.FONT_HERSHEY_SIMPLEX
    
# Define parámetros del texto
position=(20,30)
font_scale=2.0
color=(0, 0, 255)
thickness=2
# Añade texto a la imagen
cv2.putText(img_rgb, "Hola", position, font, font_scale, color, thickness)

plt.figure()
plt.axis("off")
plt.imshow(img_rgb) 
plt.show()

In [ ]:
#Para más tipografías, usar PIL como alternativa
img = cv2.imread('mandril.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Conversión de numpy array a PIL Image (tener presente que debe ser RGB, por ello la conversión previa)
pil_image = Image.fromarray(img_rgb) 
#Crea objeto para dibujar
draw = ImageDraw.Draw(pil_image)

# Carga tipografía (puede haber errores) y define parámetros
position=(20,30)
font_size=50.0
color=(0, 0, 255)
#Tipografías en Windows
font = ImageFont.truetype("arial.ttf", font_size)
#font = ImageFont.truetype("times.ttf", font_size)
#font = ImageFont.truetype("verdana.ttf", font_size)
#font = ImageFont.truetype("tahoma.ttf", font_size)

# Añade texto a la imagen
draw.text(position, "Hola", fill=color, font=font)

# Convierte resultado a numpy array para matplotlib
final_img_rgb = np.array(pil_image)

plt.figure()
plt.axis("off")
plt.imshow(final_img_rgb) 
plt.show()

Convierte la imagen a grises para procesamiento posterior

In [ ]:
#Conversión de la imagen a niveles de grises de la imagen original en BGR
gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print(gris.shape)
#Muestra, indicando el mapa de color de grises con matplotlib
plt.figure()
#Eliminamos etiquetas de los ejes
plt.axis("off")
#Muestra la imegen en grises
plt.imshow(gris, cmap='gray') 
plt.show()


Canny, detector de contornos multietapa. Descrito en mayor detalle en las sesiones de teoría (tema 4)

In [ ]:
#Obtiene contornos con el operador de Canny
#Parámetros: imagen de entrada, umbral inferior, umbral superior
canny = cv2.Canny(gris, 100, 200) #prueba a jugar con umbrales (entre 0 y 255)
#print(canny) #Muestra contenido, valores 0 o 255
#Visualiza resultado
plt.imshow(canny, cmap='gray') 
plt.show()


Cuenta el número de píxeles no nulos por columna y visualiza el vector resultante

In [ ]:
#El contenido de la imagen resultado de Canny, son valores 0 o 255, lo compruebas al descomentar
#print(canny)
#Cuenta el número de píxeles blancos (255) por columna
#Suma los valores de los pixeles por columna
col_counts = cv2.reduce(canny, 0, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)

#Normaliza en base al número de filas, primer valor devuelto por shape, y al valor máximo del píxel (255)
#El resultado será el número de píxeles blancos por columna
cols = col_counts[0] / (255 * canny.shape[0])

#Muestra dicha cuenta gráficamente
plt.figure()
plt.subplot(1, 2, 1)
plt.axis("off")
plt.title("Canny")
plt.imshow(canny, cmap='gray') 

plt.subplot(1, 2, 2)
plt.title("Respuesta de Canny")
plt.xlabel("Columnas")
plt.ylabel("% píxeles")
plt.plot(cols)
#Rango en x definido por las columnas
plt.xlim([0, canny.shape[1]])

TAREA: Realiza la cuenta de píxeles blancos por filas (en lugar de por columnas). Determina el valor máximo de píxeles blancos para filas, maxfil, mostrando el número de filas y sus respectivas posiciones, con un número de píxeles blancos mayor o igual que 0.90*maxfil. Resalta con alguna primitiva gráfica en la imagen de Canny las filas que cumplen dicha condición.

**Solución TAREA 1:** conteo de píxeles blancos por filas sobre la imagen de Canny, cálculo de `maxfil` y selección de las filas con al menos `0.90*maxfil` píxeles blancos.

In [ ]:
# Algunas imágenes tienen un marco/borde en las últimas filas o columnas que Canny detecta como un "borde"
# larguísimo y falsea el máximo; se ignora un pequeño margen de MARGEN píxeles en todo el contorno
MARGEN = 3
canny_m = canny.copy()
canny_m[:MARGEN, :] = 0; canny_m[-MARGEN:, :] = 0
canny_m[:, :MARGEN] = 0; canny_m[:, -MARGEN:] = 0

# Cuenta de píxeles blancos (255) por fila: se suma por filas (eje 1) y se divide entre 255
row_counts = cv2.reduce(canny_m, 1, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)
filas = (row_counts[:, 0] / 255).astype(int)       # nº de píxeles blancos de cada fila

# Valor máximo y umbral del 90 %
maxfil = filas.max()
umbral_fil = 0.90 * maxfil

# Filas que cumplen la condición (>= 0.90*maxfil)
filas_sel = np.where(filas >= umbral_fil)[0]

print(f"maxfil = {maxfil} píxeles blancos (en la fila {np.argmax(filas)})")
print(f"Umbral 0.90*maxfil = {umbral_fil:.1f}")
print(f"Nº de filas que cumplen la condición: {len(filas_sel)}")
print(f"Posiciones (filas): {filas_sel}")

# Resalta las filas seleccionadas sobre la imagen de Canny (convertida a 3 canales para poder dibujar en color)
canny_color = cv2.cvtColor(canny, cv2.COLOR_GRAY2RGB)
for f in filas_sel:
    cv2.line(canny_color, (0, int(f)), (canny.shape[1] - 1, int(f)), (255, 0, 0), 1)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.axis("off")
plt.title("Canny con filas destacadas")
plt.imshow(canny_color)

# Perfil de píxeles blancos por fila (eje y invertido para alinear con la imagen)
plt.subplot(1, 2, 2)
plt.title("Píxeles blancos por fila")
plt.xlabel("# píxeles blancos")
plt.ylabel("Filas")
plt.plot(filas, np.arange(len(filas)))
plt.axvline(umbral_fil, color='r', linestyle='--', label='0.90*maxfil')
plt.ylim([len(filas) - 1, 0])
plt.legend()
plt.tight_layout()
plt.show()

Canny es el detector de bordes más destacado antes de los modelos de aprendizaje profundo. Un detector más simple es Sobel (realmente suele usarse como primer paso de Canny).
El operador de Sobel considera que cuando hay un borde, el valor de intensidad de los píxeles cercanos cambia de forma notable, calcular las derivadas proporciona una evidencia de dicho cambio. El operador de Sobel aproxima el cálculo de la derivada aplicando un kernel de tamaño impar basado en el patrón [1 2 1].

In [ ]:
# Gaussiana para suavizar la imagen original, eliminando altas frecuencias
ggris = cv2.GaussianBlur(gris, (3, 3), 0)

#Calcula en ambas direcciones (horizontal y vertical)
sobelx = cv2.Sobel(ggris, cv2.CV_64F, 1, 0)  # x
sobely = cv2.Sobel(ggris, cv2.CV_64F, 0, 1)  # y
#Combina ambos resultados
sobel = cv2.add(sobelx, sobely)

#Muestra ambos resultados
plt.figure()
plt.subplot(1, 3, 1)
plt.axis("off")
plt.title('Bordes verticales')
#Verticales
#Para visualizar convierte a escala manejable en una imagen de grises
plt.imshow(cv2.convertScaleAbs(sobelx), cmap='gray') 
#plt.imshow(sobelx, cmap='gray') #Prueba sin convertir escala

plt.subplot(1, 3, 2)
plt.axis("off")
plt.title('Bordes horizontales')
#Horizontales
#Para visualizar convierte a escala manejable en una imagen de grises
plt.imshow(cv2.convertScaleAbs(sobely), cmap='gray') 
#plt.imshow(sobelx, cmap='gray') #Prueba sin convertir escala

plt.subplot(1, 3, 3)
plt.axis("off")
plt.title('Bordes combinados')
#Para visualizar convierte a escala manejable en una imagen de grises
plt.imshow(cv2.convertScaleAbs(sobel), cmap='gray') 
#plt.imshow(sobel, cmap='gray') #Prueba sin convertir escala
plt.show()

Observa que visualizar la imagen resultado de Sobel directamente como escala de grises produce un resultado diferente. Ha sido necesario convertir a datos de tipo byte. La siguiente celda muestra dos variantes de conversión, con OpenCV y numpy, evidenciando alguna diferencia, además de los valores antes y tras escalar a 8 bits

In [ ]:
# Mostrar el tipo de dato de los valores en la imagen soble, además de valores máximo y mínimo
print(f"Tipo de datos, valor mínimo y máximo en sobel: {sobel.dtype}, {np.min(sobel)}, {np.max(sobel)}")

# Conversión a byte con openCV
sobel8 = cv2.convertScaleAbs(sobel)
# Mostrar el tipo de dato de los valores en la imagen soble, además de valores máximo y mínimo
print(f"Tipo de datos, valor mínimo y máximo en sobel8: {sobel8.dtype}, {np.min(sobel8)}, {np.max(sobel8)}")

# Conversión a byte con numpy
sobel8np = np.uint8(np.abs(sobel))
# Mostrar el tipo de dato de los valores en la imagen soble, además de valores máximo y mínimo
print(f"Tipo de datos, valor mínimo y máximo en sobel8np: {sobel8np.dtype}, {np.min(sobel8np)}, {np.max(sobel8np)}")

plt.figure()
plt.subplot(1, 3, 1)
plt.axis("off")
plt.title('convertScaleAbs')
plt.imshow(sobel8, cmap='gray') 

plt.subplot(1, 3, 2)
plt.axis("off")
plt.title('np.uint8(np.abs())')
plt.imshow(sobel8np, cmap='gray') 

plt.subplot(1, 3, 3)
plt.axis("off")
plt.title('Diferencias')
plt.imshow(cv2.absdiff(sobel8,sobel8np), cmap='gray') 

plt.show()

Umbralizado de una imagen

In [ ]:
#Define valor umbral
valorUmbral = 130 #Prueba otros valores
#Obtiene imagen umbralizada para dicho valor definido
_, imagenUmbralizada = cv2.threshold(gris, valorUmbral, 255, cv2.THRESH_BINARY)
#Muestra resultado
plt.imshow(imagenUmbralizada, cmap='gray')
plt.show()

#El umbralizado inverso, los valores menores que el umbral se muestran en blanco
_, imagenUmbralizada = cv2.threshold(gris, valorUmbral, 255, cv2.THRESH_BINARY_INV)
#Muestra resultado
plt.imshow(imagenUmbralizada, cmap='gray')
plt.show()

#Valores en rango determinado por dos umbrales
imagenEnRango = cv2.inRange(gris, 150, 200)
#Muestra resultado
plt.imshow(imagenEnRango, cmap='gray')
plt.show()

El histograma de una imagen aporta información sobre el valor de umbral a elegir para ciertas situaciones

In [ ]:
#Cálculo del histograma de una imagen en escala de grises
hist = cv2.calcHist([gris], [0], None, [256], [0, 256])

plt.figure()
plt.axis("off")
plt.imshow(gris, cmap='gray')

# Histograma sin normalizar
plt.figure()
plt.subplot(1, 2, 1)
plt.title("Histograma")
plt.xlabel("Bins")
plt.ylabel("# píxeles")
plt.plot(hist)
plt.xlim([0, 256])

#Normaliza el histograma en base al número de píxeles y lo muestra
hist /= hist.sum()

plt.subplot(1, 2, 2)
plt.title("Histograma")
plt.xlabel("Bins")
plt.ylabel("% píxeles")
plt.tight_layout(pad=3.0) #separación entre plots
plt.plot(hist)
plt.xlim([0, 256])

TAREA: Aplica umbralizado a la imagen resultante de Sobel (convertida a 8 bits), y posteriormente realiza el conteo por filas y columnas similar al realizado en el ejemplo con la salida de Canny de píxeles no nulos. Calcula el valor máximo de la cuenta por filas y columnas, y determina las filas y columnas por encima del 0.90*máximo. Remarca con alguna primitiva gráfica dichas filas y columnas sobre la imagen del mandril. Visualiza los resultados obtenidos para la imagen (o una de tu elección) con Canny y Sobe tras umbralizar ¿Cómo se comparan los resultados obtenidos a partir de Sobel y Canny?

**Solución TAREA 2:** se define una función que, dada una imagen binaria, cuenta los píxeles no nulos por filas y columnas, calcula los máximos y devuelve las filas y columnas por encima del `0.90*máximo`. Se aplica a la salida de **Sobel umbralizada** y a la de **Canny** para poder compararlas.

In [ ]:
def analiza_filas_columnas(binaria, frac=0.90, margen=3):
    """Cuenta píxeles no nulos por fila y columna de una imagen binaria (0/255).
    Se ignora un margen en el contorno para que un posible marco de la imagen no falsee el máximo.
    Devuelve los conteos, los máximos y los índices de filas/columnas con conteo >= frac*máximo."""
    binaria = binaria.copy()
    binaria[:margen, :] = 0; binaria[-margen:, :] = 0
    binaria[:, :margen] = 0; binaria[:, -margen:] = 0
    filas = (cv2.reduce(binaria, 1, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)[:, 0] / 255).astype(int)
    cols = (cv2.reduce(binaria, 0, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)[0, :] / 255).astype(int)
    max_f, max_c = filas.max(), cols.max()
    filas_sel = np.where(filas >= frac * max_f)[0]
    cols_sel = np.where(cols >= frac * max_c)[0]
    return filas, cols, max_f, max_c, filas_sel, cols_sel


def resalta(img_rgb_base, filas_sel, cols_sel, color_f=(255, 0, 0), color_c=(0, 255, 0)):
    """Dibuja líneas sobre una copia de la imagen: filas en rojo y columnas en verde."""
    salida = img_rgb_base.copy()
    h, w = salida.shape[:2]
    for f in filas_sel:
        cv2.line(salida, (0, int(f)), (w - 1, int(f)), color_f, 1)
    for c in cols_sel:
        cv2.line(salida, (int(c), 0), (int(c), h - 1), color_c, 1)
    return salida


# Imagen base (el mandril en RGB, sin el texto dibujado antes)
base_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Sobel en 8 bits (ya calculado antes) y umbralizado
valor_umbral_sobel = 100   # prueba otros valores
_, sobel_bin = cv2.threshold(sobel8, valor_umbral_sobel, 255, cv2.THRESH_BINARY)

# Análisis para Sobel umbralizado y para Canny
resultados = {}
for nombre, binaria in [("Sobel umbralizado", sobel_bin), ("Canny", canny)]:
    resultados[nombre] = analiza_filas_columnas(binaria)
    filas, cols, max_f, max_c, filas_sel, cols_sel = resultados[nombre]
    print(f"--- {nombre} ---")
    print(f"Píxeles no nulos totales: {np.count_nonzero(binaria)}")
    print(f"Máximo por filas: {max_f} | filas >= 0.90*max: {len(filas_sel)} -> {filas_sel}")
    print(f"Máximo por columnas: {max_c} | columnas >= 0.90*max: {len(cols_sel)} -> {cols_sel}")
    print()

In [ ]:
# Visualización: binaria con líneas, mandril con líneas y perfiles de conteo
plt.figure(figsize=(14, 9))
for i, (nombre, binaria) in enumerate([("Sobel umbralizado", sobel_bin), ("Canny", canny)]):
    filas, cols, max_f, max_c, filas_sel, cols_sel = resultados[nombre]

    # Imagen binaria (en color para poder resaltar)
    plt.subplot(2, 4, 4 * i + 1)
    plt.axis("off")
    plt.title(nombre)
    plt.imshow(resalta(cv2.cvtColor(binaria, cv2.COLOR_GRAY2RGB), filas_sel, cols_sel))

    # Mandril con filas (rojo) y columnas (verde) resaltadas
    plt.subplot(2, 4, 4 * i + 2)
    plt.axis("off")
    plt.title(f"Mandril + {nombre}")
    plt.imshow(resalta(base_rgb, filas_sel, cols_sel))

    # Perfil por filas
    plt.subplot(2, 4, 4 * i + 3)
    plt.title("Píxeles por fila")
    plt.plot(filas, np.arange(len(filas)))
    plt.axvline(0.90 * max_f, color='r', linestyle='--')
    plt.ylim([len(filas) - 1, 0])

    # Perfil por columnas
    plt.subplot(2, 4, 4 * i + 4)
    plt.title("Píxeles por columna")
    plt.plot(cols)
    plt.axhline(0.90 * max_c, color='g', linestyle='--')
    plt.xlim([0, len(cols) - 1])

plt.tight_layout()
plt.show()

**¿Cómo se comparan Sobel y Canny?** (resultados obtenidos con la imagen estándar del mandril de 512x512 y `valor_umbral_sobel = 100`; con otra imagen u otro umbral los números cambiarán, pero la interpretación general se mantiene)

- **Cantidad y grosor de los bordes:** con ese umbral ambos métodos dejan un número parecido de píxeles no nulos (≈62 000 con Sobel frente a ≈61 000 con Canny), pero están distribuidos de forma distinta. Sobel umbralizado produce bordes *gruesos* y agrupados (varios píxeles de ancho) porque responde a toda la transición de intensidad, mientras que Canny, gracias a la supresión de no máximos, da bordes de *1 píxel* y mucho más finos y conectados.
- **Dependencia de parámetros:** Sobel depende de un solo umbral (y de la conversión a 8 bits): bajo, llena la imagen de textura y ruido (el pelaje); alto, rompe los contornos. Canny usa histéresis con dos umbrales, conservando los bordes débiles conectados a bordes fuertes y descartando los aislados, por lo que es más robusto.
- **Filas seleccionadas:** coinciden en lo esencial. Ambos métodos concentran las filas con más contornos en la parte superior de la cara: la frente/pelaje superior (primeras filas) y la zona de las cejas y ojos (filas ≈80-100). Son las zonas con más textura y cambios de intensidad.
- **Columnas seleccionadas:** aquí aparecen las diferencias. Canny muestra dos lóbulos casi simétricos de columnas ≥ 0.90*máximo, a ambos lados de la cara (≈85-130 y ≈380-410), donde está el pelaje de las mejillas, y un perfil de conteo más equilibrado entre izquierda y derecha. Sobel solo selecciona el lóbulo izquierdo (≈97-127) y una columna aislada (≈288, un pico estrecho del perfil sobre el eje vertical del hocico); el lóbulo derecho queda por debajo del 90 % porque el máximo de Sobel está inflado por esos picos y por la iluminación, que hace que los gradientes sean menores en ese lado.
- **Sensibilidad del criterio 0.90*máximo:** al depender de un único valor máximo, un pico aislado (como el de Sobel en la columna ≈288 o el marco de la imagen, que se ha excluido con `margen`) condiciona toda la selección. Canny, con perfiles más homogéneos, resulta aquí más estable.
- **Coste:** Sobel es una simple convolución (rápido y sencillo); Canny añade suavizado, gradiente, supresión de no máximos e histéresis (más costoso, pero con mejor calidad de contorno).

*Conclusión:* Sobel es útil como detector rápido o como paso previo, pero tras umbralizar da bordes gruesos y un resultado más dependiente del umbral. Canny proporciona contornos finos y más repetibles, y las filas/columnas que selecciona describen mejor dónde se concentra la información de borde de la imagen.

Diferencia de imágenes

In [ ]:
#Calcula la diferencia entre dos imágenes
#Utiliza la imagen original y la obtenida tras aplicar la gaussiana (creada en la celda dedicada a Sobel)
dif = cv2.absdiff(gris, ggris)

#Visualiza
plt.figure()
plt.subplot(1, 2, 1)
plt.title("Diferencias")
plt.axis("off")
plt.imshow(dif, cmap='gray') 

#Zonas de mayor diferencia tras aplicar umbral
res, imgdif = cv2.threshold(dif, 30, 255, cv2.THRESH_BINARY)
#Visualiza
plt.subplot(1, 2, 2)
plt.title("Mayores")
plt.axis("off")
plt.imshow(imgdif, cmap='gray') 
plt.show()


Webcam y sustracción de fotogramas

In [ ]:
vid = cv2.VideoCapture(0)

#Marca de inicio
disponible = 0 
while(True):      
    # fotograma a fotograma
    ret, frame = vid.read()

    if ret:
        if disponible > 0:
            dif = cv2.absdiff(frame, pframe)        
            # Muestra resultado
            cv2.imshow('Diferencia', dif)    
        else:
            disponible = 1

        #Copia fotograma actual para la diferencia en el siguiente forograma
        pframe = frame.copy()
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()

Webcam y sustracción de modelo del fondo

In [ ]:
vid = cv2.VideoCapture(0)

# Fondo
# Inicializa la sustracción del fondo con mezcla de gaussianas y detección de sombras
eliminadorFondo = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=50, detectShadows=True)
  
while(True):      
    # fotograma a fotograma
    ret, frame = vid.read()

    if ret:
        # Aplica efecto espejo sobre la entrada
        framem=cv2.flip(frame, 1)
        
        #Con un segundo parámerto se puede definir máscara con zonas a actualizar
        objetos = eliminadorFondo.apply(framem)
        #objetos = eliminadorFondo.apply(framem, objetos, 0)  #No actualiza el fondo
        # Obtiene fondo
        background = eliminadorFondo.getBackgroundImage()

        # Muestra resultado
        cv2.imshow('Fotograma', objetos)
        # Muestra fondo
        cv2.imshow('Fondo', background)
  
   
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()

TAREA: Tras ver los vídeos [My little piece of privacy](https://www.niklasroy.com/project/88/my-little-piece-of-privacy), [Messa di voce](https://youtu.be/GfoqiyB1ndE?feature=shared) y [Virtual air guitar](https://youtu.be/FIAmyoEpV5c?feature=shared) proponer un demostrador reinterpretando la parte de procesamiento de la imagen, tomando como punto de partida alguna de dichas instalaciones.

## Solución TAREA 3: demostrador «Arpa virtual» (*Air Strings*)

**Instalaciones de partida.** *Virtual Air Guitar* convierte el movimiento de las manos frente a una cámara en música, *Messa di voce* usa la silueta de los intérpretes para generar gráficos reactivos y *My little piece of privacy* es una instalación interactiva donde una cámara y un mecanismo físico responden a las personas que pasan frente a la ventana.

**Reinterpretación.** Se propone una guitarra/arpa virtual sin guantes de colores ni hardware especial, usando únicamente técnicas de esta práctica:

1. **Modelo de fondo (MOG2)** para segmentar a la persona (o su mano) frente a la cámara, con efecto espejo.
2. **Umbralizado** de la máscara para eliminar las sombras (valor 127 en MOG2) y **apertura morfológica** para quitar ruido.
3. La imagen se divide en **N bandas horizontales = cuerdas**. En cada banda se **cuentan los píxeles no nulos** de la máscara (la misma idea que el conteo por filas/columnas de Canny y Sobel).
4. Si el porcentaje de píxeles en movimiento de una banda supera un umbral (flanco de subida) la cuerda se **«pulsa»**: se dibuja vibrando (onda sinusoidal amortiguada, como en *Messa di voce* donde el gráfico nace de la silueta) y se muestra el nombre de la nota.

El procesamiento está encapsulado en la clase `ArpaVirtual`, independiente de la fuente de vídeo, de forma que se puede probar con la webcam, con un fichero de vídeo o con la simulación sintética incluida (útil si no hay cámara).

In [ ]:
class ArpaVirtual:
    """Cuerdas virtuales activadas por el movimiento detectado con sustracción de fondo."""

    NOTAS = ["Mi", "La", "Re", "Sol", "Si", "Mi agudo"]

    def __init__(self, n_cuerdas=6, umbral_activacion=0.06, alto_banda=0.5):
        self.n = n_cuerdas
        self.umbral = umbral_activacion      # fracción mínima de píxeles en movimiento en la banda
        self.alto_banda = alto_banda         # alto de la banda respecto al espacio entre cuerdas
        self.amplitud = np.zeros(self.n)     # amplitud de vibración actual de cada cuerda
        self.activa_prev = np.zeros(self.n, dtype=bool)
        self.t = 0
        self.mog = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=50, detectShadows=True)
        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    def procesar(self, frame):
        """Devuelve (imagen de salida BGR, máscara de movimiento, lista de notas pulsadas)."""
        framem = cv2.flip(frame, 1)                          # efecto espejo
        h, w = framem.shape[:2]

        # 1) Segmentación: MOG2 -> quita sombras (127) -> apertura morfológica
        mascara = self.mog.apply(framem)
        _, mascara = cv2.threshold(mascara, 200, 255, cv2.THRESH_BINARY)
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, self.kernel)

        # 2) Posición de las cuerdas (equiespaciadas) y alto de cada banda
        ys = np.linspace(h * 0.15, h * 0.85, self.n).astype(int)
        paso = ys[1] - ys[0]
        semialto = max(2, int(paso * self.alto_banda / 2))

        # 3) Conteo de píxeles no nulos por banda y detección de pulsación (flanco de subida)
        pulsadas = []
        for i, y in enumerate(ys):
            banda = mascara[max(0, y - semialto):y + semialto, :]
            ratio = cv2.countNonZero(banda) / banda.size
            activa = ratio > self.umbral
            if activa and not self.activa_prev[i]:
                self.amplitud[i] = 1.0
                pulsadas.append(self.NOTAS[i % len(self.NOTAS)])
            self.activa_prev[i] = activa

        # 4) Dibujo: máscara teñida de fondo + cuerdas vibrando
        salida = framem.copy()
        tinte = np.zeros_like(salida)
        tinte[mascara > 0] = (0, 255, 255)
        salida = cv2.addWeighted(salida, 1.0, tinte, 0.35, 0)

        x = np.arange(w)
        for i, y in enumerate(ys):
            a = self.amplitud[i]
            onda = y + a * (paso * 0.35) * np.sin(2 * np.pi * x / (w / (3 + i)) + self.t * 0.8)
            puntos = np.stack([x, onda], axis=1).astype(np.int32).reshape(-1, 1, 2)
            color = (0, int(255 - 120 * a), int(120 + 135 * a))
            cv2.polylines(salida, [puntos], False, color, 2 if a < 0.2 else 3)
            cv2.putText(salida, self.NOTAS[i % len(self.NOTAS)], (8, y - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        self.amplitud *= 0.92            # amortiguación de la vibración
        self.t += 1
        return salida, mascara, pulsadas

In [ ]:
# Simulación sintética (sin webcam): una "mano" (círculo) cruza las cuerdas de arriba a abajo
def fotograma_sintetico(t, ancho=640, alto=480):
    rng = np.random.default_rng(t)
    fondo = np.full((alto, ancho, 3), 90, np.uint8)
    cv2.rectangle(fondo, (0, alto // 2), (ancho, alto), (70, 80, 60), -1)        # "suelo" estático
    fondo = cv2.add(fondo, rng.integers(0, 4, fondo.shape, dtype=np.uint8))      # ruido ligero
    if t >= 30:                                                                   # tras 30 fotogramas aparece la mano
        y = int(60 + (t - 30) * 9) % alto
        cv2.circle(fondo, (ancho // 2, y), 45, (180, 150, 200), -1)
    return fondo

arpa = ArpaVirtual()
capturas = {}
for t in range(90):
    salida, mascara, notas = arpa.procesar(fotograma_sintetico(t))
    if notas:
        print(f"Fotograma {t}: nota(s) {notas}")
    if t in (30, 50, 62, 80):
        capturas[t] = salida

plt.figure(figsize=(14, 4))
for i, (t, im) in enumerate(capturas.items()):
    plt.subplot(1, len(capturas), i + 1)
    plt.axis("off")
    plt.title(f"Fotograma {t}")
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
plt.tight_layout()
plt.show()

In [ ]:
# Demostrador en vivo con webcam (ESC para salir). Para usar un vídeo: FUENTE = 'video.mp4'
FUENTE = 0

vid = cv2.VideoCapture(FUENTE)
if not vid.isOpened():
    print("No se pudo abrir la fuente de vídeo; usa la simulación sintética de la celda anterior.")
else:
    arpa = ArpaVirtual()
    while True:
        ret, frame = vid.read()
        if not ret:
            break
        salida, mascara, notas = arpa.procesar(frame)
        for nota in notas:
            print("♪", nota)              # aquí se podría lanzar un sonido (p. ej. con simpleaudio / pygame)
        cv2.imshow('Arpa virtual', salida)
        cv2.imshow('Mascara de movimiento', mascara)
        if cv2.waitKey(20) == 27:        # ESC
            break
    vid.release()
    cv2.destroyAllWindows()